In [ ]:
import sys
import time
from dotenv import load_dotenv

sys.path.insert(0, '..')
load_dotenv('../.env')

from retrieval.dense_retriever import DenseRetriever
from retrieval.sparse_retriever import SparseRetriever
from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import Reranker

COLLECTION = 'tiangolo_fastapi_ast'
QUERY = 'How does FastAPI handle dependency injection?'

print('Retrievers loaded')

In [ ]:
dense = DenseRetriever(COLLECTION, persist_dir='../chroma_db')

t0 = time.perf_counter()
dense_results = dense.retrieve(QUERY, top_k=10)
dense_latency = (time.perf_counter() - t0) * 1000

print(f'Dense retrieval: {dense_latency:.0f}ms')
for i, r in enumerate(dense_results[:3]):
    print(f'  [{i+1}] {r["metadata"]["file_path"]} (score: {r["score"]:.3f})')
    print(f'       {r["text"][:100]}...')

In [ ]:
sparse = SparseRetriever(COLLECTION, bm25_dir='../bm25_indexes')

t0 = time.perf_counter()
sparse_results = sparse.retrieve(QUERY, top_k=10)
sparse_latency = (time.perf_counter() - t0) * 1000

print(f'Sparse retrieval: {sparse_latency:.0f}ms')
for i, r in enumerate(sparse_results[:3]):
    print(f'  [{i+1}] {r["metadata"]["file_path"]} (score: {r["score"]:.3f})')
    print(f'       {r["text"][:100]}...')

In [ ]:
hybrid = HybridRetriever(COLLECTION, persist_dir='../chroma_db', bm25_dir='../bm25_indexes')
reranker = Reranker(top_k=5)

t0 = time.perf_counter()
hybrid_results = hybrid.retrieve(QUERY, top_k=10)
reranked = reranker.rerank(QUERY, hybrid_results)
total_latency = (time.perf_counter() - t0) * 1000

print(f'Hybrid + rerank: {total_latency:.0f}ms')
for i, r in enumerate(reranked):
    print(
        f'  [{i+1}] {r["metadata"]["file_path"]} '
        f'(rerank: {r["rerank_score"]:.3f}, rrf: {r["rrf_score"]:.4f})'
    )
    print(f'       Fn: {r["metadata"].get("node_name", "N/A")}')
    print()

In [ ]:
import matplotlib.pyplot as plt

alphas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
top_files = []

# Note: HybridRetriever loads the BM25 index on each instantiation.
for alpha in alphas:
    h = HybridRetriever(
        COLLECTION,
        persist_dir='../chroma_db',
        bm25_dir='../bm25_indexes',
        alpha=alpha,
    )
    results = h.retrieve(QUERY, top_k=5)
    top_files.append([r['metadata']['file_path'] for r in results])

print('Top file per alpha value:')
for alpha, files in zip(alphas, top_files):
    print(f'  alpha={alpha:.1f}: {files[0] if files else "N/A"}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar([str(a) for a in alphas], [len(f) for f in top_files], color='steelblue')
ax.set_xlabel('Alpha')
ax.set_ylabel('Results returned')
ax.set_title('Results count across alpha values')
plt.tight_layout()
plt.show()